# AI008 Training Pipeline Task Notebook (W6-T1 to W7-T8)

This notebook is a sprint execution board in notebook form.
It is intentionally **placeholder-first**: most task cells are dummy TODO blocks.


## Setup

Run this once to locate `training_pipeline` and enable imports.


In [18]:
from pathlib import Path
import sys


def find_training_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    for candidate in candidates:
        # Case 1: running from repo root (contains training_pipeline)
        tp = candidate / "training_pipeline"
        if tp.exists() and (tp / "src").exists() and (tp / "configs").exists():
            return tp

        # Case 2: running inside training_pipeline already
        if (candidate / "src").exists() and (candidate / "configs").exists():
            return candidate

        # Case 3: legacy nested path heuristic
        legacy = candidate / "ai-ml" / "training_pipeline"
        if legacy.exists() and (legacy / "src").exists():
            return legacy

    raise FileNotFoundError("Could not locate training_pipeline root")


def repo_rel(path: Path) -> str:
    resolved = path.resolve()
    parts = resolved.parts
    if "ai-ml" in parts:
        return str(Path(*parts[parts.index("ai-ml"):]))
    return str(path)


TRAINING_ROOT = find_training_root(Path.cwd())
SRC_DIR = TRAINING_ROOT / "src"

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("TRAINING_ROOT:", repo_rel(TRAINING_ROOT))


TRAINING_ROOT: ai-ml\training_pipeline


## Task Index

W6: T1, T2, T3, T4, T5, T6, T7, T8  
W7: T1, T2, T3, T4, T5, T6, T7, T8


## W6-T1

**Objective:** Project scaffold and runtime directories  
**Status:** Runnable Baseline


In [19]:
from src.utils.paths import ensure_runtime_dirs, CONFIGS_DIR, LOGS_DIR, CHECKPOINTS_DIR

ensure_runtime_dirs()
print("Configs:", repo_rel(CONFIGS_DIR))
print("Logs:", repo_rel(LOGS_DIR))
print("Checkpoints:", repo_rel(CHECKPOINTS_DIR))


Configs: ai-ml\training_pipeline\configs
Logs: ai-ml\training_pipeline\logs
Checkpoints: ai-ml\training_pipeline\checkpoints


## W6-T2

**Objective:** Configuration loading and schema validation  
**Status:** Runnable Baseline


In [20]:
from pprint import pprint
from src.core.config_manager import load_config

cfg_path = TRAINING_ROOT / "configs" / "default_config.yaml"
cfg = load_config(cfg_path)
print("Config loaded from:", repo_rel(cfg_path))

sections = ["dataset", "preprocessing", "model", "training", "output"]
for section in sections:
    print(f"\n[{section}]")
    pprint(cfg.get(section, {}), sort_dicts=False)

# Useful resolved paths (repo-relative)
dataset_cfg = cfg.get("dataset", {})
output_cfg = cfg.get("output", {})
if dataset_cfg.get("path"):
    print("\nResolved dataset.path:", repo_rel((TRAINING_ROOT / dataset_cfg["path"]).resolve()))
if dataset_cfg.get("abnormal_path"):
    print("Resolved dataset.abnormal_path:", repo_rel((TRAINING_ROOT / dataset_cfg["abnormal_path"]).resolve()))
if output_cfg.get("path"):
    print("Resolved output.path:", repo_rel((TRAINING_ROOT / output_cfg["path"]).resolve()))
if output_cfg.get("log_path"):
    print("Resolved output.log_path:", repo_rel((TRAINING_ROOT / output_cfg["log_path"]).resolve()))


Config loaded from: ai-ml\training_pipeline\configs\default_config.yaml

[dataset]
{'path': 'data/raw/example.csv',
 'abnormal_path': 'data/raw/abnormal.csv',
 'target_column': 'label',
 'train_split': 0.7,
 'val_split': 0.15,
 'test_split': 0.15,
 'random_seed': 42,
 'stratify': True}

[preprocessing]
{'missing_value_strategy': 'mean',
 'normalization': 'standard',
 'encoding': 'onehot',
 'feature_selection': True}

[model]
{'type': 'random_forest',
 'hyperparameters': {'n_estimators': 100, 'max_depth': 10}}

[training]
{'batch_size': 32, 'epochs': 50, 'learning_rate': 0.001}

[output]
{'path': 'checkpoints/',
 'log_path': 'logs/',
 'save_best_only': True,
 'checkpoint_prefix': 'model'}

Resolved dataset.path: ai-ml\training_pipeline\data\raw\example.csv
Resolved dataset.abnormal_path: ai-ml\training_pipeline\data\raw\abnormal.csv
Resolved output.path: ai-ml\training_pipeline\checkpoints
Resolved output.log_path: ai-ml\training_pipeline\logs


## W6-T3

**Objective:** Dataset loading + train/validation/test splitter  
**Status:** Runnable Baseline


In [21]:
try:
    from src.data.dataset_loader import DatasetLoader
    from src.data.splitter import DatasetSplitter
except ModuleNotFoundError as exc:
    splits = None
    print("Skipped W6-T3 smoke run because a dependency is missing:", exc)
else:
    main_path = TRAINING_ROOT / "data" / "raw" / "example.csv"
    main_dataset, _ = DatasetLoader.load_main_and_abnormal(main_path)

    X, y = DatasetLoader.separate_features_and_target(main_dataset.data, target_column="label")
    splits = DatasetSplitter.split(X, y, test_size=0.2, val_size=0.2, random_seed=42, stratify=True)

    print("Dataset:", main_dataset.name, "rows=", len(main_dataset.data))
    print("Split sizes:", len(splits.x_train), len(splits.x_val), len(splits.x_test))
    print("Label columns present:", splits.y_train is not None and splits.y_val is not None and splits.y_test is not None)


Dataset: main_dataset rows= 20
Split sizes: 12 4 4
Label columns present: True


## W6-T4

**Objective:** Preprocessing and feature loader integration  
**Status:** TODO


In [22]:
from src.preprocessing.preprocess import preprocess_features

raw_df = main_dataset.data.copy() if "main_dataset" in globals() else None
if raw_df is None:
    print("Run W6-T3 first to load sample data.")
else:
    cleaned_preprocessed, prep_events = preprocess_features(
        data=raw_df,
        cleaning_config={},
        preprocessing_config={
            "encoding": {"enabled": False},
            "normalization": {"enabled": True, "method": "standard"},
        },
        target_column="label",
    )

    print("Input shape:", raw_df.shape)
    print("Output shape:", cleaned_preprocessed.shape)
    print("Target column dtype (should remain discrete/int-like):", cleaned_preprocessed["label"].dtype)
    print("Preprocessing events (last 5):")
    for event in prep_events[-5:]:
        print(" -", event)


Input shape: (20, 3)
Output shape: (17, 3)
Target column dtype (should remain discrete/int-like): int64
Preprocessing events (last 5):
 - {'step': 'target_protection', 'details': 'excluded_from_transform=label'}
 - {'step': 'encoding', 'details': 'skipped_disabled'}
 - {'step': 'normalization', 'details': "method=standard; columns=['temperature', 'humidity']"}
 - {'step': 'feature_selection', 'details': 'skipped_no_selected_features'}
 - {'step': 'end', 'details': 'output_shape=(17, 3)'}


## W6-T5

**Objective:** Trainer + model registry training flow  
**Status:** Runnable Baseline


In [23]:
if "splits" not in globals() or splits is None:
    print("Run W6-T3 first (or install missing dependencies) before W6-T5.")
else:
    try:
        from src.core.trainer import TrainingConfig, GenericTrainingEngine
    except ModuleNotFoundError as exc:
        print("Skipped W6-T5 smoke run because a dependency is missing:", exc)
    else:
        train_X = splits.x_train
        train_y = splits.y_train
        test_X = splits.x_test

        config = TrainingConfig(
            model_type="sklearn",
            model_name="random_forest",
            model_params={"n_estimators": 50, "random_state": 42},
            verbose=True,
        )

        engine = GenericTrainingEngine(config)
        train_result = engine.fit(train_X, train_y)
        preds = engine.predict(test_X)

        print("Train result:", train_result)
        print("Predictions preview:", preds[:5])


Finished training sklearn model: random_forest
Train result: {'model_type': 'sklearn', 'status': 'trained'}
Predictions preview: [0 0 1 0]


## W6-T6

**Objective:** Metrics and validation pipeline  
**Status:** TODO


In [24]:
from src.evaluation.validator import validate_predictions

if "preds" not in globals() or "splits" not in globals() or splits is None:
    print("Run W6-T5 first so predictions are available.")
else:
    try:
        _metrics = validate_predictions(splits.y_test, preds)
        print("Validation metrics:", _metrics)
    except NotImplementedError as exc:
        print("W6-T6 is intentionally a placeholder in this branch:")
        print(" ", exc)


W6-T6 is intentionally a placeholder in this branch:
  W6-T6 will implement validation flow here.


## W6-T7

**Objective:** Checkpoint save/load implementation  
**Status:** TODO


In [25]:
from src.core.checkpoint_manager import CheckpointManager

if "engine" not in globals():
    print("Run W6-T5 first to train a model.")
else:
    ckpt_dir = TRAINING_ROOT / "checkpoints"
    manager = CheckpointManager(checkpoint_dir=ckpt_dir)
    ckpt_path = manager.save(engine.model, "notebook_demo_model")

    print("Checkpoint saved:", repo_rel(ckpt_path))
    loaded_obj = manager.load(ckpt_path.name)
    print("Checkpoint loaded object type:", type(loaded_obj).__name__)


Checkpoint saved: ai-ml\training_pipeline\checkpoints\notebook_demo_model.joblib


Checkpoint loaded object type: RandomForestClassifier


## W6-T8

**Objective:** Logging integration and run tracing  
**Status:** TODO


In [26]:
from src.core.logger import get_logger, log_run_event, log_metric

nb_logger = get_logger(
    name="notebook_demo",
    run_id="pipeline_notebook",
    log_dir=TRAINING_ROOT / "logs",
)

log_run_event(nb_logger, "Notebook logging demo started", section="W6-T8")
log_metric(nb_logger, "demo_metric", 0.123, step=1)
print("Logger configured. Check logs/pipeline_notebook.log for entries.")


2026-04-17 01:50:53,772 | INFO | notebook_demo | Notebook logging demo started | context={"section": "W6-T8"}
2026-04-17 01:50:53,773 | INFO | notebook_demo | METRIC | {"metric": "demo_metric", "value": 0.123, "step": 1, "epoch": null}


Logger configured. Check logs/pipeline_notebook.log for entries.


## W7-T1

**Objective:** CLI entry and run mode orchestration  
**Status:** TODO


In [27]:
from src.main import run_training_pipeline

try:
    run_result = run_training_pipeline(
        config_path=TRAINING_ROOT / "configs" / "default_config.yaml",
        run_id="notebook_full_run",
        save_checkpoint=False,
    )
    print("Pipeline run completed.")
    print("Run id:", run_result.get("run_id"))
    print("Rows summary:", run_result.get("rows"))
    print("Metrics keys:", list(run_result.get("metrics", {}).keys()))
except NotImplementedError:
    print("Evaluator is placeholder, but pipeline should handle this by skipping eval in main.")
except ModuleNotFoundError as exc:
    print("Missing dependency for full run:", exc)


2026-04-17 01:50:53,797 | INFO | training_pipeline | CONFIG SNAPSHOT | {"dataset": {"path": "data/raw/example.csv", "abnormal_path": "data/raw/abnormal.csv", "target_column": "label", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}, "preprocessing": {"missing_value_strategy": "mean", "normalization": "standard", "encoding": "onehot", "feature_selection": true}, "model": {"type": "random_forest", "hyperparameters": {"n_estimators": 100, "max_depth": 10}}, "training": {"batch_size": 32, "epochs": 50, "learning_rate": 0.001}, "output": {"path": "checkpoints/", "log_path": "logs/", "save_best_only": true, "checkpoint_prefix": "model"}}
2026-04-17 01:50:53,801 | INFO | training_pipeline | Dataset loaded | context={"main_rows": 20, "abnormal_rows": 3, "combined_rows": 23}
2026-04-17 01:50:53,809 | INFO | training_pipeline | Preprocessing complete | context={"processed_rows": 20, "processed_columns": 3, "preprocessing_events": 10}
2026-04-17 01:

Pipeline run completed.
Run id: notebook_full_run
Rows summary: {'input': 23, 'processed': 20, 'train': 14, 'val': 3, 'test': 3}
Metrics keys: ['validation', 'test']


## W7-T2

**Objective:** Data contract alignment with ingestion outputs  
**Status:** TODO


In [28]:
from src.utils.config_schema import CONFIG_SCHEMA

required_dataset_keys = [k for k, meta in CONFIG_SCHEMA["dataset"].items() if meta[1]]
print("Required dataset config keys:", required_dataset_keys)

if "main_dataset" in globals():
    expected_columns = {"label"}
    actual_columns = set(main_dataset.data.columns)
    print("Dataset columns:", sorted(actual_columns))
    print("Has required target column:", expected_columns.issubset(actual_columns))
else:
    print("Run W6-T3 first to inspect dataset columns.")


Required dataset config keys: ['path', 'train_split', 'val_split', 'test_split']
Dataset columns: ['humidity', 'label', 'temperature']
Has required target column: True


## W7-T3

**Objective:** Feature persistence and reload flow  
**Status:** TODO


In [29]:
import json

if "cleaned_preprocessed" not in globals():
    print("Run W6-T4 first to produce preprocessed features.")
else:
    artifact_dir = TRAINING_ROOT / "artifacts"
    artifact_dir.mkdir(parents=True, exist_ok=True)

    features_csv = artifact_dir / "features_snapshot.csv"
    mapping_json = artifact_dir / "feature_mapping.json"

    cleaned_preprocessed.to_csv(features_csv, index=False)
    mapping = {"columns": cleaned_preprocessed.columns.tolist(), "row_count": int(len(cleaned_preprocessed))}
    mapping_json.write_text(json.dumps(mapping, indent=2), encoding="utf-8")

    from src.preprocessing.feature_loader import load_feature_set
    reloaded = load_feature_set(features_csv, mapping_json)

    print("Saved features:", repo_rel(features_csv))
    print("Reloaded shape:", reloaded["data"].shape)
    print("Reloaded mapping keys:", reloaded["mapping"].keys())


Saved features: ai-ml\training_pipeline\artifacts\features_snapshot.csv
Reloaded shape: (17, 3)
Reloaded mapping keys: dict_keys(['columns', 'row_count'])


## W7-T4

**Objective:** Training enhancements (early stopping / scheduling)  
**Status:** TODO


In [30]:
# W7-T4 demo: simple experiment sweep using trainer configuration changes

if "splits" not in globals() or splits is None:
    print("Run W6-T3 first to create train/test splits.")
else:
    from src.core.trainer import TrainingConfig, GenericTrainingEngine

    sweep = [
        {"n_estimators": 20, "max_depth": 4, "random_state": 42},
        {"n_estimators": 100, "max_depth": 10, "random_state": 42},
    ]

    sweep_results = []
    for hp in sweep:
        cfg = TrainingConfig(
            model_type="sklearn",
            model_name="random_forest",
            model_params=hp,
            verbose=False,
        )
        model = GenericTrainingEngine(cfg)
        model.fit(splits.x_train, splits.y_train)
        pred = model.predict(splits.x_test)
        sweep_results.append({"hyperparameters": hp, "n_predictions": int(len(pred))})

    print("Sweep results:")
    for row in sweep_results:
        print(" -", row)


Sweep results:
 - {'hyperparameters': {'n_estimators': 20, 'max_depth': 4, 'random_state': 42}, 'n_predictions': 4}
 - {'hyperparameters': {'n_estimators': 100, 'max_depth': 10, 'random_state': 42}, 'n_predictions': 4}


## W7-T5

**Objective:** Evaluation report generation  
**Status:** TODO


In [31]:
import json
from datetime import datetime

report_dir = TRAINING_ROOT / "reports"
report_dir.mkdir(parents=True, exist_ok=True)
report_path = report_dir / "evaluation_report_notebook.json"

if "preds" not in globals():
    print("Run W6-T5 first to generate predictions.")
else:
    report = {
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "status": "partial",
        "notes": "W6-T6 evaluator is placeholder in this branch.",
        "prediction_count": int(len(preds)),
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("Report written:", repo_rel(report_path))
    print(report)


Report written: ai-ml\training_pipeline\reports\evaluation_report_notebook.json
{'generated_at': '2026-04-16T15:50:54.173048Z', 'status': 'partial', 'notes': 'W6-T6 evaluator is placeholder in this branch.', 'prediction_count': 4}


## W7-T6

**Objective:** Resume-from-checkpoint training flow  
**Status:** TODO


In [32]:
# W7-T6 demo: load saved model artifact and run inference (if checkpoint exists)

from src.core.checkpoint_manager import CheckpointManager

manager = CheckpointManager(checkpoint_dir=TRAINING_ROOT / "checkpoints")
candidates = sorted((TRAINING_ROOT / "checkpoints").glob("*.joblib"))

if not candidates:
    print("No .joblib checkpoints found. Run W6-T7 first.")
elif "splits" not in globals() or splits is None:
    print("Run W6-T3 first to get test features.")
else:
    latest = candidates[-1]
    loaded_model = manager.load(latest.name)
    loaded_preds = loaded_model.predict(splits.x_test)
    print("Loaded checkpoint:", repo_rel(latest))
    print("Inference preview:", loaded_preds[:5])


Loaded checkpoint: ai-ml\training_pipeline\checkpoints\notebook_demo_model.joblib
Inference preview: [0 0 1 0]


## W7-T7

**Objective:** Experiment artifact tracking  
**Status:** TODO


In [33]:
import json
from datetime import datetime

artifact_root = TRAINING_ROOT / "artifacts"
artifact_root.mkdir(parents=True, exist_ok=True)
manifest_path = artifact_root / "experiment_manifest.json"

manifest = {
    "run_id": "pipeline_notebook_demo",
    "timestamp_utc": datetime.utcnow().isoformat() + "Z",
    "files": [
        str(p.relative_to(TRAINING_ROOT)).replace('\\', '/')
        for p in [
            TRAINING_ROOT / "configs" / "default_config.yaml",
            TRAINING_ROOT / "logs",
            TRAINING_ROOT / "checkpoints",
        ]
        if p.exists()
    ],
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Manifest written:", repo_rel(manifest_path))
print(manifest)


Manifest written: ai-ml\training_pipeline\artifacts\experiment_manifest.json
{'run_id': 'pipeline_notebook_demo', 'timestamp_utc': '2026-04-16T15:50:54.233567Z', 'files': ['configs/default_config.yaml', 'logs', 'checkpoints']}


## W7-T8

**Objective:** End-to-end pipeline integration test  
**Status:** TODO


In [34]:
# W7-T8 end-to-end smoke style check in notebook

from src.main import run_training_pipeline

try:
    final_run = run_training_pipeline(
        config_path=TRAINING_ROOT / "configs" / "default_config.yaml",
        run_id="notebook_e2e",
        save_checkpoint=False,
    )
    assert "rows" in final_run
    assert "metrics" in final_run
    print("E2E run OK. Summary:", final_run["rows"])
except Exception as exc:
    print("E2E demo encountered an issue:", type(exc).__name__, str(exc))


2026-04-17 01:50:54,256 | INFO | training_pipeline | CONFIG SNAPSHOT | {"dataset": {"path": "data/raw/example.csv", "abnormal_path": "data/raw/abnormal.csv", "target_column": "label", "train_split": 0.7, "val_split": 0.15, "test_split": 0.15, "random_seed": 42, "stratify": true}, "preprocessing": {"missing_value_strategy": "mean", "normalization": "standard", "encoding": "onehot", "feature_selection": true}, "model": {"type": "random_forest", "hyperparameters": {"n_estimators": 100, "max_depth": 10}}, "training": {"batch_size": 32, "epochs": 50, "learning_rate": 0.001}, "output": {"path": "checkpoints/", "log_path": "logs/", "save_best_only": true, "checkpoint_prefix": "model"}}
2026-04-17 01:50:54,259 | INFO | training_pipeline | Dataset loaded | context={"main_rows": 20, "abnormal_rows": 3, "combined_rows": 23}
2026-04-17 01:50:54,265 | INFO | training_pipeline | Preprocessing complete | context={"processed_rows": 20, "processed_columns": 3, "preprocessing_events": 10}
2026-04-17 01:

E2E run OK. Summary: {'input': 23, 'processed': 20, 'train': 14, 'val': 3, 'test': 3}


## Completion Notes

When a task is implemented:
1. Replace the dummy code cell for that task with real pipeline code.
2. Add a short verification/assertion block.
3. Update that task status from `TODO` to `Done`.
